In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


In [0]:
dbutils.widgets.text("sql_server", "azure-server2.database.windows.net", "Azure SQL server")
dbutils.widgets.text("sql_database", "finance-database1", "Azure SQL database")
dbutils.widgets.text("secret_scope", "finance-kv", "Databricks secret scope backed by Key Vault")
dbutils.widgets.text("tenant_id", "d424bee3-0f15-41b1-bd08-7f8cd526fcec", "Azure AD tenant ID")

SQL_SERVER = dbutils.widgets.get("sql_server")
SQL_DATABASE = dbutils.widgets.get("sql_database")
SECRET_SCOPE = dbutils.widgets.get("secret_scope")
TENANT_ID = dbutils.widgets.get("tenant_id")

assert TENANT_ID, "tenant_id widget must be set — find it on the app registration's Overview page."

In [0]:
sp_client_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-client-id")
sp_client_secret = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-client-secret")

jdbc_url = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;"
    f"database={SQL_DATABASE};"
    f"encrypt=true;trustServerCertificate=false;loginTimeout=30;"
    f"authentication=ActiveDirectoryServicePrincipal;"
)

control_table = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.ingestion_metadata")
    .option("user", sp_client_id)
    .option("password", sp_client_secret)
    .load()
)

print(f"Control table rows: {control_table.count()} (expected 24 per kickoff doc)")
display(control_table)

Control table rows: 24 (expected 24 per kickoff doc)


dataset_name,asset_type,exchange_code,source_url,bronze_container,bronze_target_path,is_active,expected_min_records,load_type,created_at
equities_BER,equity,BER,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/BER.csv,bronze,equities/BER,Y,6990,full,2026-08-20T05:29:00.347Z
equities_BSE,equity,BSE,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/BSE.csv,bronze,equities/BSE,Y,3470,full,2026-08-20T05:29:00.347Z
equities_FRA,equity,FRA,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/FRA.csv,bronze,equities/FRA,Y,10431,full,2026-08-20T05:29:00.347Z
equities_GER,equity,GER,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/GER.csv,bronze,equities/GER,Y,1070,full,2026-08-20T05:29:00.347Z
equities_JPX,equity,JPX,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/JPX.csv,bronze,equities/JPX,Y,2754,full,2026-08-20T05:29:00.347Z
equities_LSE,equity,LSE,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/LSE.csv,bronze,equities/LSE,Y,3052,full,2026-08-20T05:29:00.347Z
equities_NSE,equity,NSE,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/NSE.csv,bronze,equities/NSE,Y,1737,full,2026-08-20T05:29:00.347Z
equities_NYQ,equity,NYQ,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/NYQ.csv,bronze,equities/NYQ,Y,3749,full,2026-08-20T05:29:00.347Z
equities_SHZ,equity,SHZ,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/SHZ.csv,bronze,equities/SHZ,Y,2024,full,2026-08-20T05:29:00.347Z
equities_VIE,equity,VIE,https://raw.githubusercontent.com/JerBouma/FinanceDatabase/main/database/equities/VIE.csv,bronze,equities/VIE,Y,1219,full,2026-08-20T05:29:00.347Z


In [0]:
from pyspark.sql import functions as F

audit_rows = []
for asset_class in ASSET_CLASSES:
    for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
        if asset_class not in classes:
            continue
        path = latest_snapshot_path(asset_class, exch)
        if path is None:
            audit_rows.append({"asset_class": asset_class, "exchange": exch, "check": "NO SNAPSHOT FOLDER FOUND"})
            continue
        try:
            df = (
            spark.read.option("header", True).option("inferSchema", True)
            .option("multiLine", True).option("escape", "\"")
            .csv(f"{path}/*.csv")
                )
            cols = df.columns
            row_count = df.count()
            looks_like_html = any(c.strip().lower().startswith("<!doctype") or c.strip() == "" for c in cols)
            check = "OK" if (row_count > 0 and not looks_like_html) else "SUSPECT — check content"
        except Exception as e:
            check = f"READ ERROR: {e}"
            row_count, cols = 0, []
        audit_rows.append({
            "asset_class": asset_class, "exchange": exch, "snapshot": path.split("snapshot_date=")[-1],
            "row_count": row_count, "column_count": len(cols), "check": check,
        })

audit_df = spark.createDataFrame(audit_rows)
display(audit_df)

suspect_count = audit_df.where("check != 'OK'").count()
print(f"{suspect_count} of {audit_df.count()} bronze targets flagged for review")
if suspect_count:
    display(audit_df.where("check != 'OK'"))

asset_class,check,column_count,exchange,row_count,snapshot
equities,OK,22,BER,7767,2026-08-21
equities,OK,22,BSE,3856,2026-08-21
equities,OK,22,FRA,11587,2026-08-21
equities,OK,22,GER,1189,2026-08-21
equities,OK,22,JPX,3061,2026-08-21
equities,OK,22,LSE,3391,2026-08-21
equities,OK,22,NSE,1930,2026-08-21
equities,OK,22,SHZ,2248,2026-08-21
equities,OK,22,VIE,1355,2026-08-21
equities,OK,22,NYQ,4165,2026-08-21


0 of 24 bronze targets flagged for review
